In [ ]:
# 4. Multi-Head Self-Attention: Theory and Advanced Implementation

## Theory: The Heart of Transformers

Multi-Head Self-Attention (MSA) is the core mechanism that enables Vision Transformers to model relationships between all patches simultaneously. Unlike CNNs that have limited receptive fields, MSA provides global receptive fields from the first layer.

### Mathematical Foundation

For input embeddings $\mathbf{Z} \in \mathbb{R}^{N \times D}$ where $N$ is the number of tokens and $D$ is the embedding dimension:

**Step 1: Linear Projections**
$$\mathbf{Q} = \mathbf{Z}\mathbf{W}^Q, \quad \mathbf{K} = \mathbf{Z}\mathbf{W}^K, \quad \mathbf{V} = \mathbf{Z}\mathbf{W}^V$$

**Step 2: Multi-Head Computation**
$$\text{head}_i = \text{Attention}(\mathbf{Q}_i, \mathbf{K}_i, \mathbf{V}_i) = \text{softmax}\left(\frac{\mathbf{Q}_i\mathbf{K}_i^T}{\sqrt{d_k}}\right)\mathbf{V}_i$$

**Step 3: Concatenation and Projection**
$$\text{MSA}(\mathbf{Z}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\mathbf{W}^O$$

### Key Innovations

1. **Scaled Attention**: The $\sqrt{d_k}$ scaling prevents softmax saturation
2. **Multi-Head**: Different heads can focus on different types of relationships
3. **Self-Attention**: Each token can attend to all other tokens

---

print("Implementing advanced multi-head self-attention...")

class MultiHeadSelfAttention(nn.Module):
    """
    Advanced Multi-Head Self-Attention with modern techniques:
    - Fused QKV projection for efficiency
    - Attention dropout for regularization  
    - Optional attention weight visualization
    - Gradient checkpointing support
    """
    
    def __init__(
        self, 
        embed_dim: int, 
        num_heads: int, 
        dropout: float = 0.0,
        attention_dropout: float = 0.0,
        bias: bool = True,
        fused_qkv: bool = True
    ):
        super().__init__()
        
        assert embed_dim % num_heads == 0, \
            f"embed_dim ({embed_dim}) must be divisible by num_heads ({num_heads})"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5  # 1/sqrt(d_k)
        self.fused_qkv = fused_qkv
        
        # Efficient fused QKV projection or separate projections
        if fused_qkv:
            self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=bias)
        else:
            self.q_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
            self.k_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
            self.v_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        
        # Output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        
        # Dropout layers
        self.attn_dropout = nn.Dropout(attention_dropout)
        self.proj_dropout = nn.Dropout(dropout)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights with proper scaling"""
        if self.fused_qkv:
            nn.init.xavier_uniform_(self.qkv.weight)
            if self.qkv.bias is not None:
                nn.init.constant_(self.qkv.bias, 0)
        else:
            for module in [self.q_proj, self.k_proj, self.v_proj]:
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
        
        nn.init.xavier_uniform_(self.out_proj.weight)
        if self.out_proj.bias is not None:
            nn.init.constant_(self.out_proj.bias, 0)
    
    def forward(
        self, 
        x: torch.Tensor, 
        attention_mask: Optional[torch.Tensor] = None,
        return_attention: bool = False
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """
        Forward pass of multi-head self-attention
        
        Args:
            x: Input tensor [B, N, D] where N = num_patches + 1
            attention_mask: Optional mask [B, N, N] or [N, N]
            return_attention: Whether to return attention weights
            
        Returns:
            Output tensor [B, N, D] and optionally attention weights [B, H, N, N]
        """
        B, N, D = x.shape
        
        # Generate Q, K, V
        if self.fused_qkv:
            # Fused approach: more efficient
            qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
            qkv = qkv.permute(2, 0, 3, 1, 4)  # [3, B, H, N, head_dim]
            q, k, v = qkv.unbind(0)  # Each: [B, H, N, head_dim]
        else:
            # Separate projections: more interpretable
            q = self.q_proj(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
            k = self.k_proj(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
            v = self.v_proj(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        attn_scores = (q @ k.transpose(-2, -1)) * self.scale  # [B, H, N, N]
        
        # Apply attention mask if provided
        if attention_mask is not None:
            if attention_mask.dim() == 2:
                attention_mask = attention_mask.unsqueeze(0).unsqueeze(0)  # [1, 1, N, N]
            elif attention_mask.dim() == 3:
                attention_mask = attention_mask.unsqueeze(1)  # [B, 1, N, N]
            
            attn_scores = attn_scores.masked_fill(attention_mask == 0, float('-inf'))
        
        # Apply softmax and dropout
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        
        # Apply attention to values
        out = (attn_weights @ v).transpose(1, 2).reshape(B, N, D)  # [B, N, D]
        
        # Output projection and dropout
        out = self.out_proj(out)
        out = self.proj_dropout(out)
        
        if return_attention:
            return out, attn_weights
        return out


class LayerScale(nn.Module):
    """
    LayerScale: learnable scaling factors for residual branches
    
    Introduced in "Going deeper with Image Transformers" (Touvron et al., 2021)
    Helps with training stability in very deep networks.
    """
    
    def __init__(self, dim: int, init_values: float = 1e-5):
        super().__init__()
        self.gamma = nn.Parameter(init_values * torch.ones(dim))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.gamma


class FeedForward(nn.Module):
    """
    Feed-Forward Network with modern improvements:
    - GELU activation (smoother than ReLU)
    - Dropout for regularization
    - Optional layer scaling
    """
    
    def __init__(
        self, 
        embed_dim: int, 
        hidden_dim: Optional[int] = None,
        dropout: float = 0.0,
        activation: nn.Module = nn.GELU,
        bias: bool = True
    ):
        super().__init__()
        
        hidden_dim = hidden_dim or 4 * embed_dim  # Standard 4x expansion
        
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim, bias=bias),
            activation(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim, bias=bias),
            nn.Dropout(dropout)
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize with proper scaling"""
        for module in self.net:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DropPath(nn.Module):
    """
    Stochastic Depth (Drop Path) regularization
    
    Randomly drops entire residual paths during training to:
    - Reduce overfitting
    - Enable training of very deep networks
    - Improve generalization
    """
    
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.drop_prob == 0.0:
            return x
        
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)  # (B, 1, 1, ...)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()  # Binarize
        
        # Scale by keep_prob to maintain expected value
        output = x.div(keep_prob) * random_tensor
        return output


class TransformerBlock(nn.Module):
    """
    Complete Transformer Encoder Block with modern improvements:
    - Pre-normalization (more stable than post-norm)
    - Stochastic depth for regularization
    - Optional layer scaling
    - Attention visualization support
    """
    
    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
        attention_dropout: float = 0.0,
        drop_path: float = 0.0,
        layer_scale_init: Optional[float] = None,
        norm_layer: nn.Module = nn.LayerNorm
    ):
        super().__init__()
        
        # Layer normalization
        self.norm1 = norm_layer(embed_dim)
        self.norm2 = norm_layer(embed_dim)
        
        # Multi-head self-attention
        self.attn = MultiHeadSelfAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            attention_dropout=attention_dropout
        )
        
        # Feed-forward network
        self.mlp = FeedForward(
            embed_dim=embed_dim,
            hidden_dim=int(embed_dim * mlp_ratio),
            dropout=dropout
        )
        
        # Stochastic depth
        self.drop_path1 = DropPath(drop_path) if drop_path > 0 else nn.Identity()
        self.drop_path2 = DropPath(drop_path) if drop_path > 0 else nn.Identity()
        
        # Optional layer scaling
        self.layer_scale1 = LayerScale(embed_dim, layer_scale_init) \
            if layer_scale_init else nn.Identity()
        self.layer_scale2 = LayerScale(embed_dim, layer_scale_init) \
            if layer_scale_init else nn.Identity()
    
    def forward(
        self, 
        x: torch.Tensor, 
        attention_mask: Optional[torch.Tensor] = None,
        return_attention: bool = False
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """
        Args:
            x: Input tensor [B, N, D]
            attention_mask: Optional attention mask
            return_attention: Whether to return attention weights
        """
        # Pre-norm + Multi-head self-attention + residual connection
        if return_attention:
            attn_out, attn_weights = self.attn(
                self.norm1(x), 
                attention_mask=attention_mask, 
                return_attention=True
            )
            x = x + self.drop_path1(self.layer_scale1(attn_out))
        else:
            attn_out = self.attn(self.norm1(x), attention_mask=attention_mask)
            x = x + self.drop_path1(self.layer_scale1(attn_out))
            attn_weights = None
        
        # Pre-norm + Feed-forward + residual connection
        mlp_out = self.mlp(self.norm2(x))
        x = x + self.drop_path2(self.layer_scale2(mlp_out))
        
        if return_attention:
            return x, attn_weights
        return x


print("✓ Advanced multi-head self-attention implementation completed")
print("✓ Includes modern techniques: LayerScale, DropPath, and efficient QKV projection")

# Vision Transformer (ViT): Theory and Implementation

This notebook provides a comprehensive implementation and analysis of Vision Transformers (ViT), including:

- **Mathematical foundations** of Vision Transformers
- **Advanced implementation** with modern techniques  
- **Multiple model variants** (ViT-Tiny, ViT-Small, ViT-Base, etc.)
- **Attention visualization** and interpretability tools
- **Training system** with best practices
- **Performance comparison** with CNNs

The Vision Transformer revolutionized computer vision by successfully applying the transformer architecture to image classification tasks, achieving state-of-the-art results without convolutional layers.

## Table of Contents

1. **Theory and Mathematical Foundations**
2. **Package Installation and Setup**
3. **Patch Embedding Theory and Implementation**
4. **Multi-Head Self-Attention Mechanisms**
5. **Complete Vision Transformer Architecture**
6. **Visualization and Analysis Tools**
7. **Advanced Training System**
8. **Data Loading and Augmentation**
9. **Training Execution and Model Comparison**
10. **Attention Analysis and Interpretability**

# 1. Vision Transformer: Mathematical Theory and Foundations

## Introduction

The Vision Transformer (ViT), introduced by Dosovitskiy et al. (2020), represents a paradigm shift in computer vision by directly applying the transformer architecture to image classification. Unlike convolutional neural networks (CNNs), ViT treats images as sequences of patches, enabling the model to capture long-range dependencies and global context effectively.

## Core Innovation: Images as Sequences

The key insight behind ViT is treating an image as a sequence of patches, similar to how text is treated as a sequence of words in natural language processing.

## Mathematical Formulation

### 1. Patch Embedding

**Image to Patches**: An image $\mathbf{x} \in \mathbb{R}^{H \times W \times C}$ is split into a sequence of flattened 2D patches $\mathbf{x}_p \in \mathbb{R}^{N \times (P^2 \cdot C)}$, where:
- $(H, W)$ is the resolution of the original image
- $C$ is the number of channels  
- $(P, P)$ is the resolution of each image patch
- $N = \frac{HW}{P^2}$ is the resulting number of patches

**Linear Projection**: Each flattened patch is linearly mapped to dimension $D$:

$$\mathbf{z}_0 = [\mathbf{x}_{\text{class}}; \mathbf{x}_p^1 \mathbf{E}; \mathbf{x}_p^2 \mathbf{E}; \ldots; \mathbf{x}_p^N \mathbf{E}] + \mathbf{E}_{\text{pos}}$$

where:
- $\mathbf{E} \in \mathbb{R}^{(P^2 \cdot C) \times D}$ is the trainable linear projection matrix
- $\mathbf{E}_{\text{pos}} \in \mathbb{R}^{(N+1) \times D}$ is the position embedding
- $\mathbf{x}_{\text{class}}$ is a learnable class token

### 2. Multi-Head Self-Attention (MSA)

The attention mechanism enables each patch to attend to all other patches:

$$\text{MSA}(\mathbf{z}) = \text{Concat}(\text{head}_1, \text{head}_2, \ldots, \text{head}_h) \mathbf{W}^O$$

Each attention head is computed as:

$$\text{head}_i = \text{Attention}(\mathbf{z} \mathbf{W}_i^Q, \mathbf{z} \mathbf{W}_i^K, \mathbf{z} \mathbf{W}_i^V)$$

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}}\right) \mathbf{V}$$

### 3. Transformer Encoder Block

Each transformer block applies:

$$\mathbf{z}' = \text{MSA}(\text{LN}(\mathbf{z})) + \mathbf{z}$$
$$\mathbf{z}'' = \text{MLP}(\text{LN}(\mathbf{z}')) + \mathbf{z}'$$

where:
- $\text{LN}$ is Layer Normalization
- $\text{MLP}$ is a two-layer feed-forward network with GELU activation

### 4. Classification Head

The final class token representation is used for classification:

$$\text{y} = \text{Linear}(\text{LN}(\mathbf{z}_L^0))$$

where $\mathbf{z}_L^0$ is the class token after $L$ transformer layers.

## Key Advantages

1. **Global Receptive Field**: Every patch can attend to every other patch from the first layer
2. **Scale Efficiency**: Computational complexity scales linearly with image size (not quadratically like CNNs)
3. **Transfer Learning**: Pre-trained on large datasets, then fine-tuned for specific tasks
4. **Interpretability**: Attention maps provide insight into model decision-making

## Computational Complexity

- **Standard CNN**: $O(H \cdot W \cdot K^2 \cdot C)$ where $K$ is kernel size
- **Vision Transformer**: $O(N^2 \cdot D + N \cdot D^2)$ where $N$ is number of patches

For large images, ViT can be more efficient than CNNs when $N < H \cdot W / P^2$.

## Model Variants

| Model | Patch Size | Embedding Dim | Layers | Heads | Parameters |
|-------|------------|---------------|---------|-------|------------|
| ViT-Tiny | 16×16 | 192 | 12 | 3 | 5.5M |
| ViT-Small | 16×16 | 384 | 12 | 6 | 22M |
| ViT-Base | 16×16 | 768 | 12 | 12 | 86M |
| ViT-Large | 16×16 | 1024 | 24 | 16 | 307M |
| ViT-Huge | 14×14 | 1280 | 32 | 16 | 632M |

## References

[1] Dosovitskiy, A., et al. "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale." ICLR 2021.

[2] Vaswani, A., et al. "Attention is All You Need." NeurIPS 2017.

[3] Ba, J. L., Kiros, J. R., & Hinton, G. E. "Layer Normalization." arXiv:1607.06450, 2016.

In [ ]:
# 2. Package Installation and Setup

print("Setting up Vision Transformer implementation...")

# Install required packages
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

# List of required packages
required_packages = [
    "torch>=1.9.0",           # PyTorch for deep learning
    "torchvision>=0.10.0",    # For datasets and transforms
    "matplotlib>=3.3.0",      # For plotting and visualization
    "seaborn>=0.11.0",        # For enhanced plots
    "numpy>=1.19.0",          # For numerical operations
    "scipy>=1.7.0",           # For scientific computing
    "einops>=0.4.0",          # For elegant tensor operations
    "tqdm>=4.60.0"            # For progress bars
]

print("Installing required packages...")
try:
    for package in required_packages:
        install_package(package.split(">=")[0])  # Install base package name
    print("✓ All packages installed successfully")
except Exception as e:
    print(f"Installation warning: {e}")
    print("Some packages may already be installed or unavailable")

# Import all required libraries
print("\nImporting libraries...")

# Core PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset

# Computer vision specific
from torchvision import datasets, transforms

# Scientific computing
import numpy as np
import scipy.ndimage

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
import math
import time
import warnings
from typing import Optional, Tuple, List, Union

# Tensor operations (optional)
try:
    from einops import rearrange, repeat
    from einops.layers.torch import Rearrange
    EINOPS_AVAILABLE = True
    print("✓ einops available for elegant tensor operations")
except ImportError:
    EINOPS_AVAILABLE = False
    print("Note: einops not available, using manual tensor operations")

# Configure warnings and display
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display system information
print(f"\n📋 System Information:")
print(f"✓ Python version: {sys.version.split()[0]}")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Device available: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

if torch.cuda.is_available():
    print(f"  - GPU: {torch.cuda.get_device_name(0)}")
    print(f"  - Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Set random seeds for reproducibility
print(f"\n🎲 Setting random seeds for reproducibility...")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("✓ Environment setup completed successfully!")
print("Ready to implement Vision Transformers 🚀")

In [ ]:
# 3. Patch Embedding: Theory and Implementation

## Theory: Converting Images to Sequences

The patch embedding is the foundation of Vision Transformers. It converts 2D images into 1D sequences that can be processed by transformer architectures.

### Mathematical Foundation

Given an image $\mathbf{x} \in \mathbb{R}^{H \times W \times C}$:

1. **Divide into patches**: Split into $N = \frac{HW}{P^2}$ patches of size $P \times P$
2. **Flatten patches**: Each patch becomes a vector of size $P^2 \cdot C$  
3. **Linear projection**: Map to embedding dimension $D$
4. **Add positional encoding**: Preserve spatial relationships
5. **Prepend class token**: Enable classification

### Implementation Strategy

We implement two approaches:
- **Convolution-based**: More efficient, uses stride=patch_size
- **Unfold-based**: Educational, shows explicit patch extraction

---

print("Implementing patch embedding components...")

class PatchEmbedding(nn.Module):
    """
    Efficient patch embedding using convolutional projection
    
    This approach uses a convolutional layer with kernel_size=patch_size and 
    stride=patch_size to efficiently extract and project patches.
    """
    
    def __init__(
        self, 
        img_size: int = 224, 
        patch_size: int = 16, 
        in_channels: int = 3, 
        embed_dim: int = 768,
        norm_layer: Optional[nn.Module] = None,
        flatten: bool = True,
        bias: bool = True
    ):
        super().__init__()
        
        # Handle different input formats
        img_size = (img_size, img_size) if isinstance(img_size, int) else img_size
        patch_size = (patch_size, patch_size) if isinstance(patch_size, int) else patch_size
        
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.flatten = flatten
        
        # Convolutional projection - equivalent to patch extraction + linear projection
        self.proj = nn.Conv2d(
            in_channels, embed_dim, 
            kernel_size=patch_size, 
            stride=patch_size, 
            bias=bias
        )
        
        # Optional normalization
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor [B, C, H, W]
        Returns:
            Patch embeddings [B, num_patches, embed_dim] if flatten=True
            else [B, embed_dim, H//patch_size, W//patch_size]
        """
        B, C, H, W = x.shape
        
        # Validate input dimensions
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input size ({H}×{W}) doesn't match expected size {self.img_size}"
        
        # Project patches: [B, C, H, W] -> [B, embed_dim, H//P, W//P]
        x = self.proj(x)
        
        if self.flatten:
            # Flatten spatial dimensions: [B, embed_dim, H//P, W//P] -> [B, num_patches, embed_dim]
            x = x.flatten(2).transpose(1, 2)
        
        # Apply normalization
        x = self.norm(x)
        return x


class PositionalEncoding(nn.Module):
    """
    Learnable positional encoding for maintaining spatial relationships
    
    Unlike fixed sinusoidal encodings, we use learnable embeddings that
    can adapt to the specific visual patterns in images.
    """
    
    def __init__(self, num_patches: int, embed_dim: int, dropout: float = 0.0):
        super().__init__()
        self.num_patches = num_patches
        
        # Learnable positional embeddings for num_patches + 1 (including CLS token)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)
        
        # Initialize with truncated normal distribution
        self._init_weights()
    
    def _init_weights(self):
        """Initialize positional embeddings with truncated normal distribution"""
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional embeddings to input tokens"""
        x = x + self.pos_embed
        return self.dropout(x)


class AdvancedPatchEmbedding(nn.Module):
    """
    Complete patch embedding module combining:
    - Patch extraction and projection
    - Class token addition  
    - Positional encoding
    """
    
    def __init__(
        self, 
        img_size: int = 224, 
        patch_size: int = 16, 
        in_channels: int = 3,
        embed_dim: int = 768, 
        dropout: float = 0.1, 
        norm_layer: Optional[nn.Module] = None
    ):
        super().__init__()
        
        # Patch embedding component
        self.patch_embed = PatchEmbedding(
            img_size=img_size,
            patch_size=patch_size, 
            in_channels=in_channels,
            embed_dim=embed_dim,
            norm_layer=norm_layer
        )
        
        num_patches = self.patch_embed.num_patches
        
        # Learnable class token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(num_patches, embed_dim, dropout)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize class token with truncated normal distribution"""
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input images [B, C, H, W]
        Returns:
            Token embeddings [B, num_patches + 1, embed_dim]
        """
        B = x.shape[0]
        
        # Extract and project patches: [B, C, H, W] -> [B, num_patches, embed_dim]
        x = self.patch_embed(x)
        
        # Add class token: [B, num_patches, embed_dim] -> [B, num_patches + 1, embed_dim]
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        
        # Add positional encoding
        x = self.pos_encoding(x)
        
        return x


# Utility function for visualizing patch extraction
def visualize_patch_extraction(image_tensor, patch_size=16):
    """
    Visualize how an image is divided into patches
    
    Args:
        image_tensor: Input image tensor [C, H, W] or [B, C, H, W]
        patch_size: Size of each patch
    """
    if image_tensor.dim() == 4:
        image_tensor = image_tensor[0]  # Take first image if batch
    
    C, H, W = image_tensor.shape
    
    # Create patch grid visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    if img_np.min() < 0:  # Denormalize if needed
        img_np = (img_np + 1) / 2
    img_np = np.clip(img_np, 0, 1)
    
    axes[0].imshow(img_np)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Image with patch grid
    axes[1].imshow(img_np)
    
    # Draw patch boundaries
    for i in range(0, H, patch_size):
        axes[1].axhline(y=i, color='red', linewidth=2, alpha=0.7)
    for j in range(0, W, patch_size):
        axes[1].axvline(x=j, color='red', linewidth=2, alpha=0.7)
    
    axes[1].set_title(f'Patch Grid ({patch_size}×{patch_size})')
    axes[1].axis('off')
    
    # Extract and show some patches
    num_patches_h = H // patch_size
    num_patches_w = W // patch_size
    
    # Show a few patches
    patches_to_show = min(8, num_patches_h * num_patches_w)
    patch_indices = np.random.choice(num_patches_h * num_patches_w, patches_to_show, replace=False)
    
    # Create subplot for patches
    axes[2].set_title('Sample Patches')
    axes[2].axis('off')
    
    # Create a grid to show patches
    grid_size = int(np.ceil(np.sqrt(patches_to_show)))
    patch_grid = np.zeros((grid_size * patch_size, grid_size * patch_size, 3))
    
    for idx, patch_idx in enumerate(patch_indices[:grid_size*grid_size]):
        patch_row = patch_idx // num_patches_w
        patch_col = patch_idx % num_patches_w
        
        # Extract patch
        patch = img_np[
            patch_row*patch_size:(patch_row+1)*patch_size,
            patch_col*patch_size:(patch_col+1)*patch_size
        ]
        
        # Place in grid
        grid_row = idx // grid_size
        grid_col = idx % grid_size
        patch_grid[
            grid_row*patch_size:(grid_row+1)*patch_size,
            grid_col*patch_size:(grid_col+1)*patch_size
        ] = patch
    
    axes[2].imshow(patch_grid)
    
    plt.tight_layout()
    plt.show()
    
    print(f"✨ Patch Information:")
    print(f"  - Image size: {H}×{W}")
    print(f"  - Patch size: {patch_size}×{patch_size}")
    print(f"  - Number of patches: {num_patches_h * num_patches_w}")
    print(f"  - Patches per row: {num_patches_w}")
    print(f"  - Patches per column: {num_patches_h}")


print("✓ Patch embedding implementation completed")
print("✓ Ready to process images as sequences of patches")

In [ ]:
# Cell 4: Complete Vision Transformer Implementation
print("Creating complete Vision Transformer with multiple variants...")

class VisionTransformer(nn.Module):
    """
    Vision Transformer (ViT) implementation with support for multiple variants:
    - ViT-Tiny, ViT-Small, ViT-Base, ViT-Large, ViT-Huge
    - Advanced training techniques (stochastic depth, layer scale)
    - Flexible input resolution and patch sizes
    - Attention visualization capabilities
    """
    
    def __init__(self, img_size: int = 224, patch_size: int = 16, in_channels: int = 3,
                 num_classes: int = 1000, embed_dim: int = 768, depth: int = 12,
                 num_heads: int = 12, mlp_ratio: float = 4.0, dropout: float = 0.0,
                 attention_dropout: float = 0.0, drop_path_rate: float = 0.0,
                 layer_scale_init: Optional[float] = None, class_token: bool = True,
                 global_pool: str = 'token'):
        super().__init__()
        
        self.num_classes = num_classes
        self.global_pool = global_pool
        self.num_features = embed_dim
        self.embed_dim = embed_dim
        self.num_prefix_tokens = 1 if class_token else 0
        
        # Patch embedding
        self.patch_embed = AdvancedPatchEmbedding(
            img_size=img_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim,
            dropout=dropout
        )
        
        num_patches = self.patch_embed.patch_embed.num_patches
        
        # Stochastic depth decay rule
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
                attention_dropout=attention_dropout,
                drop_path=dpr[i],
                layer_scale_init=layer_scale_init
            ) for i in range(depth)
        ])
        
        # Final layer norm
        self.norm = nn.LayerNorm(embed_dim)
        
        # Classification head
        self.head = nn.Linear(embed_dim, num_classes) if num_classes > 0 else nn.Identity()
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using truncated normal distribution"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.bias, 0)
                nn.init.constant_(m.weight, 1.0)
    
    def forward_features(self, x: torch.Tensor, return_all_tokens: bool = False,
                        return_attention: bool = False) -> torch.Tensor:
        """
        Forward pass through transformer layers
        
        Args:
            x: Input images [B, C, H, W]
            return_all_tokens: Return all patch tokens instead of just class token
            return_attention: Return attention weights from all layers
        """
        x = self.patch_embed(x)  # [B, num_patches + 1, embed_dim]
        
        attention_weights = []
        
        for block in self.blocks:
            if return_attention:
                x, attn = block(x, return_attention=True)
                attention_weights.append(attn)
            else:
                x = block(x)
        
        x = self.norm(x)
        
        if return_attention:
            if return_all_tokens:
                return x, attention_weights
            return x[:, 0], attention_weights  # Return class token
        
        if return_all_tokens:
            return x
        return x[:, 0]  # Return class token
    
    def forward(self, x: torch.Tensor, return_attention: bool = False) -> torch.Tensor:
        """
        Complete forward pass
        
        Args:
            x: Input images [B, C, H, W]
            return_attention: Return attention weights along with predictions
        """
        if return_attention:
            x, attention_weights = self.forward_features(x, return_attention=True)
            x = self.head(x)
            return x, attention_weights
        else:
            x = self.forward_features(x)
            x = self.head(x)
            return x

# Predefined ViT configurations
def create_vit_configs():
    """Create standard ViT model configurations"""
    configs = {
        'vit_tiny': {
            'patch_size': 16, 'embed_dim': 192, 'depth': 12, 'num_heads': 3
        },
        'vit_small': {
            'patch_size': 16, 'embed_dim': 384, 'depth': 12, 'num_heads': 6
        },
        'vit_base': {
            'patch_size': 16, 'embed_dim': 768, 'depth': 12, 'num_heads': 12
        },
        'vit_large': {
            'patch_size': 16, 'embed_dim': 1024, 'depth': 24, 'num_heads': 16
        },
        'vit_huge': {
            'patch_size': 14, 'embed_dim': 1280, 'depth': 32, 'num_heads': 16
        }
    }
    return configs

def create_model(model_name: str, num_classes: int = 10, img_size: int = 32,
                 drop_path_rate: float = 0.1, **kwargs) -> VisionTransformer:
    """
    Create a ViT model with predefined configuration
    
    Args:
        model_name: Name of the model variant
        num_classes: Number of output classes
        img_size: Input image size
        drop_path_rate: Stochastic depth rate
        **kwargs: Additional arguments
    """
    configs = create_vit_configs()
    
    if model_name not in configs:
        raise ValueError(f"Unknown model: {model_name}. Available: {list(configs.keys())}")
    
    config = configs[model_name].copy()
    config.update(kwargs)
    
    model = VisionTransformer(
        img_size=img_size,
        num_classes=num_classes,
        drop_path_rate=drop_path_rate,
        **config
    )
    
    return model

# Create different model variants for comparison
print("Creating ViT model variants...")

# Small models for CIFAR-10 (32x32 images)
vit_tiny_cifar = create_model('vit_tiny', num_classes=10, img_size=32, patch_size=4)
vit_small_cifar = create_model('vit_small', num_classes=10, img_size=32, patch_size=4)

# Print model information
def print_model_info(model, name):
    """Print model parameter count and architecture info"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"\n{name}:")
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Model size: {total_params * 4 / 1024**2:.2f} MB")

print_model_info(vit_tiny_cifar, "ViT-Tiny (CIFAR-10)")
print_model_info(vit_small_cifar, "ViT-Small (CIFAR-10)")

print("✓ Complete Vision Transformer implementation created")

# Vision Transformer (ViT) Mathematical Formulation

The paper "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale" by Alexey Dosovitskiy et al. introduces the Vision Transformer (ViT) for image classification [1]. Here's the key mathematical formulation of the Vision Transformer:

## Patch Embedding

1. **Image to Patches**: An image $\mathbf{x} \in \mathbb{R}^{H \times W \times C}$ is split into a sequence of flattened 2D patches $\mathbf{x}_p \in \mathbb{R}^{N \times (P^2 \cdot C)}$, where $(H, W)$ is the resolution of the original image, $C$ is the number of channels, $(P, P)$ is the resolution of each image patch, and $N = \frac{HW}{P^2}$ is the resulting number of patches.

2. **Linear Projection of Flattened Patches**: Each flattened patch is linearly mapped to a vector of dimension $D$, which corresponds to the input dimension of the transformer:

   $\mathbf{z}_0 = [\mathbf{x}_p^1 \mathbf{E}; \mathbf{x}_p^2 \mathbf{E}; \ldots; \mathbf{x}_p^N \mathbf{E}] + \mathbf{E}_{\text{pos}}$

   where $\mathbf{E} \in \mathbb{R}^{(P^2 \cdot C) \times D}$ is a trainable linear projection and $\mathbf{E}_{\text{pos}} \in \mathbb{R}^{N \times D}$ is the position embedding.

## Transformer Encoder

The transformer encoder consists of alternating layers of multi-head self-attention (MSA) and multi-layer perceptron (MLP) blocks [2].

3. **Multi-Head Self-Attention (MSA)**:

   $\text{MSA}(\mathbf{z}) = \text{Concat}(\text{head}_1, \text{head}_2, \ldots, \text{head}_h) \mathbf{W}^O$

   where each head is computed as:

   $\text{head}_i = \text{Attention}(\mathbf{z} \mathbf{W}_i^Q, \mathbf{z} \mathbf{W}_i^K, \mathbf{z} \mathbf{W}_i^V)$

   and the attention function is:

   $\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}}\right) \mathbf{V}$

   Here, $\mathbf{W}_i^Q, \mathbf{W}_i^K, \mathbf{W}_i^V \in \mathbb{R}^{D \times d_k}$ and $\mathbf{W}^O \in \mathbb{R}^{h \cdot d_k \times D}$ are trainable weight matrices, $d_k$ is the dimension of each head, and $h$ is the number of heads.

4. **Multi-Layer Perceptron (MLP)**:

   $\text{MLP}(\mathbf{z}) = \text{GELU}(\mathbf{z} \mathbf{W}_1 + \mathbf{b}_1) \mathbf{W}_2 + \mathbf{b}_2$

   where $\mathbf{W}_1 \in \mathbb{R}^{D \times D_{\text{MLP}}}$, $\mathbf{W}_2 \in \mathbb{R}^{D_{\text{MLP}} \times D}$, $\mathbf{b}_1 \in \mathbb{R}^{D_{\text{MLP}}}$, and $\mathbf{b}_2 \in \mathbb{R}^{D}$ are trainable parameters, and GELU is the Gaussian Error Linear Unit activation function [3].

5. **Layer Normalization and Residual Connections**:
   Each block includes layer normalization (LN) and residual connections [4]:

   $\mathbf{z}' = \text{MSA}(\text{LN}(\mathbf{z})) + \mathbf{z}$

   $\mathbf{z}'' = \text{MLP}(\text{LN}(\mathbf{z}')) + \mathbf{z}'$

## Output Layer

6. **Classification Head**:
   The final representation $\mathbf{z}_L^0$ (corresponding to the class token) is used for classification:

   $\text{y} = \text{MLP}(\mathbf{z}_L^0)$

   where $\mathbf{z}_L^0$ is the class token representation after the $L$-th layer.

## Summary of the Forward Pass

1. Convert the image to patches and project them to obtain the initial patch embeddings.
2. Pass the patch embeddings through the transformer encoder, which consists of $L$ layers of MSA and MLP blocks.
3. Use the output corresponding to the class token for classification through an MLP head.

This summarizes the key components and mathematical formulation of the Vision Transformer as presented in the paper.

References:

[1] Dosovitskiy, A., Beyer, L., Kolesnikov, A., Weissenborn, D., Zhai, X., Unterthiner, T., ... & Houlsby, N. (2020). An image is worth 16x16 words: Transformers for image recognition at scale. arXiv preprint arXiv:2010.11929.

[2] Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., ... & Polosukhin, I. (2017). Attention is all you need. Advances in neural information processing systems, 30.

[3] Hendrycks, D., & Gimpel, K. (2016). Gaussian error linear units (gelus). arXiv preprint arXiv:1606.08415.

[4] Ba, J. L., Kiros, J. R., & Hinton, G. E. (2016). Layer normalization. arXiv preprint arXiv:1607.06450.

In [ ]:
# Cell 2: Advanced Patch Embedding Implementation
print("Creating enhanced patch embedding implementation...")

class PatchEmbedding(nn.Module):
    """
    Advanced patch embedding module with multiple implementation strategies
    
    Converts image into sequence of patch embeddings with proper initialization
    and optional techniques like patch dropout and learnable position embeddings.
    """
    
    def __init__(self, img_size: int = 224, patch_size: int = 16, in_channels: int = 3, 
                 embed_dim: int = 768, norm_layer: Optional[nn.Module] = None, 
                 flatten: bool = True, bias: bool = True):
        super().__init__()
        
        img_size = (img_size, img_size) if isinstance(img_size, int) else img_size
        patch_size = (patch_size, patch_size) if isinstance(patch_size, int) else patch_size
        
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.flatten = flatten
        
        # Convolutional projection (more efficient than unfold + linear)
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, 
                             stride=patch_size, bias=bias)
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor [B, C, H, W]
        Returns:
            Patch embeddings [B, num_patches, embed_dim] if flatten=True
            else [B, embed_dim, H//patch_size, W//patch_size]
        """
        B, C, H, W = x.shape
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input size ({H}x{W}) doesn't match expected size {self.img_size}"
        
        x = self.proj(x)  # [B, embed_dim, H//patch_size, W//patch_size]
        
        if self.flatten:
            x = x.flatten(2).transpose(1, 2)  # [B, num_patches, embed_dim]
        
        x = self.norm(x)
        return x

class PositionalEncoding(nn.Module):
    """
    Learnable positional encoding with optional interpolation for different sizes
    """
    
    def __init__(self, num_patches: int, embed_dim: int, dropout: float = 0.0):
        super().__init__()
        self.num_patches = num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)
        
        # Initialize positional embeddings
        self._init_weights()
    
    def _init_weights(self):
        """Initialize positional embeddings with truncated normal"""
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional embeddings to input"""
        return self.dropout(x + self.pos_embed)

class AdvancedPatchEmbedding(nn.Module):
    """
    Complete patch embedding with CLS token and positional encoding
    """
    
    def __init__(self, img_size: int = 224, patch_size: int = 16, in_channels: int = 3,
                 embed_dim: int = 768, dropout: float = 0.1, norm_layer: Optional[nn.Module] = None):
        super().__init__()
        
        self.patch_embed = PatchEmbedding(
            img_size=img_size, patch_size=patch_size, in_channels=in_channels,
            embed_dim=embed_dim, norm_layer=norm_layer
        )
        
        num_patches = self.patch_embed.num_patches
        
        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(num_patches, embed_dim, dropout)
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize CLS token"""
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input images [B, C, H, W]
        Returns:
            Embeddings with CLS token [B, num_patches + 1, embed_dim]
        """
        B = x.shape[0]
        
        # Extract patches
        x = self.patch_embed(x)  # [B, num_patches, embed_dim]
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)  # [B, num_patches + 1, embed_dim]
        
        # Add positional encoding
        x = self.pos_encoding(x)
        
        return x

print("✓ Advanced patch embedding implementation created")